# Notebook 04: Neural Feature Extraction

**Purpose:** Extract spectrograms for neural network training

**Key Tasks:**
1. Compute log-mel spectrograms (1-channel)
2. Compute delta and delta-delta (3-channel)
3. Normalize spectrograms
4. Save arrays for CNN training

**Outputs:**
- spectrograms_1ch.npy (N, 99, 40, 1)
- spectrograms_3ch.npy (N, 99, 40, 3)

---

## 1. Import Libraries

In [1]:
import os, sys
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import librosa
import librosa.feature
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from tqdm import tqdm

print('✓ Libraries imported')

✓ Libraries imported


## 2. Project Setup

In [2]:
PROJECT_ROOT = Path('/Users/harryirving/Development/projects/ai-ml/BikeAIv5')
os.chdir(PROJECT_ROOT)

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'universal'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'neural'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
LOGS_DIR = PROJECT_ROOT / 'logs'

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Features Output: {FEATURES_DIR}')

Features Output: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/neural


## 3. Setup MLflow

In [3]:
mlflow.set_tracking_uri(f'file://{LOGS_DIR / "mlruns"}')
mlflow.set_experiment('angle_grinder_pipeline')
print('✓ MLflow configured')

✓ MLflow configured


## 4. Load Manifest

In [4]:
manifest_path = PROJECT_ROOT / 'data' / 'processed' / 'manifest.json'

if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest not found. Run Notebook 02 first.')

with open(manifest_path, 'r') as f:
    manifest = json.load(f)

df_manifest = pd.DataFrame(manifest)
print(f'✓ Loaded {len(df_manifest)} segments')
print(df_manifest['class'].value_counts())

✓ Loaded 60348 segments
class
grinder       32938
tools         19341
background     8069
Name: count, dtype: int64


## 5. Define Spectrogram Extraction

In [ ]:
def extract_log_mel_spectrogram(audio, sr=44100, n_mels=40, n_fft=512, hop_length=160):
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel.T

def extract_3channel_spectrogram(audio, sr=44100, n_mels=40, n_fft=512, hop_length=160):
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    delta = librosa.feature.delta(log_mel, order=1)
    delta2 = librosa.feature.delta(log_mel, order=2)
    return np.stack([log_mel.T, delta.T, delta2.T], axis=-1)

def normalize_spectrogram(spec):
    mean, std = np.mean(spec), np.std(spec)
    return (spec - mean) / std if std > 0 else spec - mean

print('✓ Functions defined')

✓ Functions defined


## 6. Extract Spectrograms

In [ ]:
print('Extracting spectrograms...')
spectrograms_1ch_list = []
spectrograms_3ch_list = []
labels_list = []
failed = 0

for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc='Extracting'):
    filepath = PROJECT_ROOT / row['filepath']
    if not filepath.exists():
        failed += 1
        continue
    try:
        audio, sr = librosa.load(filepath, sr=44100)
        spec_1ch = normalize_spectrogram(extract_log_mel_spectrogram(audio, sr))
        spec_3ch = normalize_spectrogram(extract_3channel_spectrogram(audio, sr))
        spectrograms_1ch_list.append(spec_1ch)
        spectrograms_3ch_list.append(spec_3ch)
        labels_list.append(row['label'])
    except Exception as e:
        failed += 1

print(f'✓ Extracted {len(spectrograms_1ch_list)} spectrograms')
if failed > 0: print(f'⚠️  {failed} failed')

Extracting spectrograms...


Extracting: 100%|██████████| 60348/60348 [03:06<00:00, 323.22it/s]

✓ Extracted 60348 spectrograms


## 7. Convert to Arrays

In [8]:
X_spec_1ch = np.expand_dims(np.array(spectrograms_1ch_list), axis=-1)
X_spec_3ch = np.array(spectrograms_3ch_list)
y = np.array(labels_list)

print(f'1-channel: {X_spec_1ch.shape}')
print(f'3-channel: {X_spec_3ch.shape}')
print(f'Labels: {y.shape}')

for label, count in zip(*np.unique(y, return_counts=True)):
    name = 'grinder' if label == 1 else 'non-grinder'
    print(f'{name}: {count} ({count/len(y)*100:.1f}%)')

1-channel: (60348, 101, 40, 1)
3-channel: (60348, 101, 40, 3)
Labels: (60348,)
non-grinder: 27410 (45.4%)
grinder: 32938 (54.6%)


## 8. Visualize Examples

In [9]:
grinder_idx = np.where(y == 1)[0][0] if np.any(y == 1) else None
non_grinder_idx = np.where(y == 0)[0][0] if np.any(y == 0) else None

if grinder_idx and non_grinder_idx:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    axes[0,0].imshow(X_spec_1ch[grinder_idx,:,:,0].T, aspect='auto', origin='lower', cmap='viridis')
    axes[0,0].set_title('Grinder - 1-Ch (Log-Mel)')
    
    axes[0,1].imshow(X_spec_3ch[grinder_idx,:,:,0].T, aspect='auto', origin='lower', cmap='viridis')
    axes[0,1].set_title('Grinder - Ch0 (Log-Mel)')
    
    axes[0,2].imshow(X_spec_3ch[grinder_idx,:,:,1].T, aspect='auto', origin='lower', cmap='viridis')
    axes[0,2].set_title('Grinder - Ch1 (Delta)')
    
    axes[1,0].imshow(X_spec_1ch[non_grinder_idx,:,:,0].T, aspect='auto', origin='lower', cmap='viridis')
    axes[1,0].set_title('Non-Grinder - 1-Ch')
    
    axes[1,1].imshow(X_spec_3ch[non_grinder_idx,:,:,0].T, aspect='auto', origin='lower', cmap='viridis')
    axes[1,1].set_title('Non-Grinder - Ch0')
    
    axes[1,2].imshow(X_spec_3ch[non_grinder_idx,:,:,1].T, aspect='auto', origin='lower', cmap='viridis')
    axes[1,2].set_title('Non-Grinder - Ch1')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR/'04_spectrogram_examples.png', dpi=300)
    print(f'✓ Saved visualization')
    plt.show()

## 9. Save Arrays

In [ ]:
np.save(FEATURES_DIR / 'spectrograms_1ch.npy', X_spec_1ch)
print(f'✓ Saved 1-ch: {X_spec_1ch.shape} ({X_spec_1ch.nbytes/1024**2:.1f} MB)')

np.save(FEATURES_DIR / 'spectrograms_3ch.npy', X_spec_3ch)
print(f'✓ Saved 3-ch: {X_spec_3ch.shape} ({X_spec_3ch.nbytes/1024**2:.1f} MB)')

np.save(FEATURES_DIR / 'labels.npy', y)
print(f'✓ Saved labels: {y.shape}')

metadata = {
    'timestamp': datetime.now().isoformat(),
    'total_samples': len(y),
    'spec_1ch_shape': list(X_spec_1ch.shape),
    'spec_3ch_shape': list(X_spec_3ch.shape),
    'label_distribution': {int(k): int(v) for k,v in zip(*np.unique(y, return_counts=True))},
    'parameters': {'sr': 44100, 'n_mels': 40, 'n_fft': 512, 'hop_length': 160}
}
with open(FEATURES_DIR / 'spectrogram_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✓ Saved metadata')

✓ Saved 1-ch: (60348, 101, 40, 1) (930.0 MB)
✓ Saved 3-ch: (60348, 101, 40, 3) (2790.1 MB)
✓ Saved labels: (60348,)
✓ Saved metadata


## 10. Log to MLflow

In [ ]:
with mlflow.start_run(run_name='04_neural_feature_extraction'):
    mlflow.log_param('sr', 44100)
    mlflow.log_param('n_mels', 40)
    mlflow.log_param('n_fft', 512)
    mlflow.log_param('hop_length', 160)
    mlflow.log_metric('total_samples', len(y))
    mlflow.log_metric('grinder_samples', int(np.sum(y==1)))
    mlflow.log_metric('non_grinder_samples', int(np.sum(y==0)))
    mlflow.log_metric('failed_extractions', failed)
    mlflow.set_tags({'stage': 'feature_extraction', 'notebook': '04', 'feature_type': 'neural'})
    mlflow.log_artifact(str(FEATURES_DIR / 'spectrogram_metadata.json'))
    if (FIGURES_DIR/'04_spectrogram_examples.png').exists():
        mlflow.log_artifact(str(FIGURES_DIR/'04_spectrogram_examples.png'))
    print('✓ Logged to MLflow')

✓ Logged to MLflow


## 11. Summary

In [12]:
print('='*70)
print('NEURAL FEATURE EXTRACTION COMPLETE')
print('='*70)
print(f'✓ Extracted {len(y)} spectrograms')
print(f'  1-ch: {X_spec_1ch.shape} ({X_spec_1ch.nbytes/1024**2:.1f} MB)')
print(f'  3-ch: {X_spec_3ch.shape} ({X_spec_3ch.nbytes/1024**2:.1f} MB)')
print(f'Saved to: {FEATURES_DIR}')
print('Ready for:')
print('  → 07_custom_cnn_training.ipynb')
print('  → 08_yamnet_transfer_learning.ipynb')
print('='*70)
print('✓ Notebook 04 Complete!')

NEURAL FEATURE EXTRACTION COMPLETE
✓ Extracted 60348 spectrograms
  1-ch: (60348, 101, 40, 1) (930.0 MB)
  3-ch: (60348, 101, 40, 3) (2790.1 MB)
Saved to: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/neural
Ready for:
  → 07_custom_cnn_training.ipynb
  → 08_yamnet_transfer_learning.ipynb
✓ Notebook 04 Complete!
